# Yuda 5 — Text Features Head-Only + Ensemble
C: Text Features (freeze encoder, head-only 10 epoch) + E: Ensemble (ConvNeXt + C soft voting)

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader

import open_clip
import timm

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_dataloaders, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


In [2]:
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}

# CLIP model & transforms
CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP visual dim: {clip_dim}")

# ConvNeXt transforms
conv_train_tfm = get_transforms(augment=True, img_size=224)
conv_val_tfm = get_transforms(augment=False, img_size=224)

results = []

CLIP visual dim: 512


In [3]:
# Full dataset
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

# CLIP val loader
class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

clip_train_ds = SingleTransformDataset(df_train, clip_transform)
clip_val_ds = SingleTransformDataset(df_val, clip_transform)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# ConvNeXt val loader
conv_val_ds = SingleTransformDataset(df_val, conv_val_tfm)
conv_val_loader = DataLoader(conv_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

Train: 21221 | Val: 5306


---
## C: Text Features (Head-Only)
Freeze CLIP encoder, train classifier 10 epoch

In [4]:
class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        prompts = ["recyclable waste", "electronic waste", "organic waste"]
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True

In [5]:
def train_head_only(model, train_loader, val_loader, name, epochs=10):
    model = model.to(DEVICE)
    model.freeze_encoder()
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_f1, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  C E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"  E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    model.load_state_dict(best_state)
    result = {'name': name, 'best_val_f1': best_f1}
    torch.save(best_state, RESULTS / f'{name}.pt')
    json.dump(result, open(RESULTS / f'{name}.json', 'w'))
    return model, best_f1

In [6]:
print("=" * 50)
print("C: Text Features (Head-Only, 10 epoch)")
print("=" * 50)

model_c = CLIPWithTextFeatures(clip_model, clip_visual, num_classes=3)
model_c, best_f1 = train_head_only(model_c, clip_train_loader, clip_val_loader, 'yuda5_C_headonly')
print(f"C Macro F1: {best_f1:.4f}")
results.append({'name': 'C_headonly', 'best_val_f1': best_f1, 'f1_per_class': []})

C: Text Features (Head-Only, 10 epoch)


  E1: val_f1=0.9699 (best=0.9699)


  E2: val_f1=0.9762 (best=0.9762)


  E3: val_f1=0.9765 (best=0.9765)


  E4: val_f1=0.9795 (best=0.9795)


...

  E6: val_f1=0.9829 (best=0.9829)


  E7: val_f1=0.9829 (best=0.9829)


  E8: val_f1=0.9835 (best=0.9835)


  E9: val_f1=0.9826 (best=0.9835)


  E10: val_f1=0.9820 (best=0.9835)
C Macro F1: 0.9835


---
## E: Ensemble (ConvNeXt + C soft voting)
Average probabilities dari kedua model

In [7]:
# Load ConvNeXt
convnext = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
ckpt = torch.load(RESULTS / 'yuda_convnext_tiny.pt', map_location=DEVICE, weights_only=True)
convnext.load_state_dict(ckpt)
convnext.eval()
print("ConvNeXt loaded")

ConvNeXt loaded


In [8]:
print("=" * 50)
print("E: Ensemble (C + ConvNeXt)")
print("=" * 50)

val_labels = np.array([_CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])

# C1 probs on val
model_c.eval()
c1_probs = []
for x, _ in tqdm(clip_val_loader, desc="C on val"):
    x = x.to(DEVICE)
    with torch.no_grad():
        out = model_c(x)
    c1_probs.append(torch.softmax(out, dim=1).cpu().numpy())
c1_probs = np.concatenate(c1_probs)
print(f"C probs: {c1_probs.shape}")

del model_c
torch.cuda.empty_cache()

# ConvNeXt probs on val
conv_probs = []
for x, _ in tqdm(conv_val_loader, desc="ConvNeXt on val"):
    x = x.to(DEVICE)
    with torch.no_grad():
        out = convnext(x)
    conv_probs.append(torch.softmax(out, dim=1).cpu().numpy())
conv_probs = np.concatenate(conv_probs)
print(f"ConvNeXt probs: {conv_probs.shape}")

# Try different weights
for w1, w2 in [(0.5, 0.5), (0.7, 0.3), (0.3, 0.7), (0.8, 0.2), (0.2, 0.8)]:
    ensemble = w1 * c1_probs + w2 * conv_probs
    preds = ensemble.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    f1_per = f1_score(val_labels, preds, average=None)
    print(f"  C={w1:.1f} ConvNeXt={w2:.1f}: F1={f1:.4f} | per-class={f1_per}")
    results.append({'name': f'E_ensemble_{w1}_{w2}', 'best_val_f1': f1, 'f1_per_class': f1_per.tolist()})

E: Ensemble (C + ConvNeXt)


C on val: 100%|██████████| 166/166 [00:12<00:00, 13.69it/s]


C probs: (5306, 3)


ConvNeXt on val: 100%|██████████| 166/166 [00:18<00:00,  9.09it/s]

ConvNeXt probs: (5306, 3)
  C=0.5 ConvNeXt=0.5: F1=0.9784 | per-class=[0.9701344  0.98671727 0.97825653]
  C=0.7 ConvNeXt=0.3: F1=0.9849 | per-class=[0.97827715 0.99305117 0.98328025]
  C=0.3 ConvNeXt=0.7: F1=0.9676 | per-class=[0.95665172 0.97683156 0.96940612]
  C=0.8 ConvNeXt=0.2: F1=0.9838 | per-class=[0.97654691 0.99305117 0.98187612]
  C=0.2 ConvNeXt=0.8: F1=0.9647 | per-class=[0.95352324 0.97253433 0.96805112]


---
## Summary

In [9]:
summary = pd.DataFrame(results)
summary = summary.sort_values('best_val_f1', ascending=False)
summary.to_csv(RESULTS / 'yuda5_summary.csv', index=False)
summary

,name,best_val_f1,f1_per_class
2,E_ensemble_0.7_0.3,0.984870,"[0.9782771535580524, 0.9930511686670878, 0.983..."
4,E_ensemble_0.8_0.2,0.983825,"[0.9765469061876247, 0.9930511686670878, 0.981..."
0,C_headonly,0.983511,[]
1,E_ensemble_0.5_0.5,0.978369,"[0.9701343952215032, 0.9867172675521821, 0.978..."
3,E_ensemble_0.3_0.7,0.967630,"[0.9566517189835575, 0.9768315591734502, 0.969..."
5,E_ensemble_0.2_0.8,0.964703,"[0.9535232383808095, 0.9725343320848939, 0.968..."
